# Markov por Estudante com Eixos Fixos (StudyChat)
Este notebook gera **heatmaps comparáveis entre alunos**, usando um **conjunto global fixo** de estados e superestados:

- `states` (1ª ordem): **todas** as labels observadas no dataset, ordenadas globalmente (mesmo se o aluno não passou por algumas).

- `superstates` (2ª ordem): **todos os pares (a,b)** observados no dataset (mesmo se um aluno não tiver percorrido algum par).

Assim, todos os usuários têm **mesmo shape e mesma ordem** nos eixos, permitindo comparação direta.


Saídas em `/mnt/data/markov_user_fixed_plots` e um JSON com a ordem global de eixos.



In [1]:

# ============================================================
# 1) Imports e setup
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from itertools import tee
from datasets import load_dataset

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.5f}')

out_dir = "artifacts"
os.makedirs(out_dir, exist_ok=True)

def pairwise(iterable):
    a, b = tee(iterable)
    next(b, None)
    return zip(a, b)

def tripletwise(iterable):
    a, b, c = tee(iterable, 3)
    next(b, None)
    next(c, None); next(c, None)
    return zip(a, b, c)

print("Diretório de saída:", out_dir)


/opt/homebrew/Caskroom/miniconda/base/envs/studychat/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Diretório de saída: artifacts


In [2]:

# ============================================================
# 2) Carregar dataset e preparar labels
# ============================================================
hf_token = os.environ.get("HF_TOKEN", None)
ds = load_dataset("wmcnicho/StudyChat", token=hf_token)

frames = []
for split in ds.keys():
    frames.append(pd.DataFrame(ds[split]))
raw_df = pd.concat(frames, ignore_index=True)

def extract_label(llm_label_entry):
    if isinstance(llm_label_entry, dict):
        return llm_label_entry.get("label")
    return None

df = raw_df.copy()
df["label"] = df["llm_label"].apply(extract_label)
df = df.dropna(subset=["label"]).copy()

# Normalização de tempo para ordenar cada chat
def safe_parse_timestamp(x):
    try:
        return int(x)  # epoch ms
    except Exception:
        pass
    try:
        return pd.to_datetime(x).value // 10**6
    except Exception:
        return np.nan

df["timestamp_ms"] = df["timestamp"].apply(safe_parse_timestamp) if "timestamp" in df.columns else np.nan
df["interactionCount"] = pd.to_numeric(df.get("interactionCount", np.nan), errors="coerce")

df["_order_key"] = list(zip(df.get("timestamp_ms", pd.Series([np.nan]*len(df))).fillna(np.inf),
                            df.get("interactionCount", pd.Series([np.nan]*len(df))).fillna(np.inf)))
df = df.sort_values(by=["chatId", "_order_key"]).drop(columns=["_order_key"])

# Sequências por chat
seq_by_chat = (
    df.groupby(["userId", "chatId"])["label"]
      .apply(list)
      .reset_index(name="sequence")
)

print("Conversas válidas:", len(seq_by_chat))
seq_by_chat.head(3)


Conversas válidas: 2214


,userId,chatId,sequence
0,011bb520-7041-704f-b3e7-ab5c43dd3950,00078d7c-022c-429b-9e07-33459b7a5f6e,"[conceptual_questions>Programming Language, co..."
1,011bb520-7041-704f-b3e7-ab5c43dd3950,0fb51040-c70d-415e-9a90-9d6766f90ffc,[writing_request>Code/Data Conversion]
2,011bb520-7041-704f-b3e7-ab5c43dd3950,10f45c05-44af-4afb-ac8c-e8fe0411e9a5,"[editing_request>Edit English, writing_request..."


In [3]:

# ============================================================
# 3) Definir eixos GLOBAIS: states (1ª ordem) e superstates (2ª ordem)
# ============================================================
# states: todas as labels observadas
global_states = sorted(df["label"].dropna().unique().tolist())
state_index = {s: i for i, s in enumerate(global_states)}

# superstates: todos os pares consecutivos observados no dataset (sem produto cartesiano completo para evitar explosão)
super_set = set()
for seq in seq_by_chat["sequence"]:
    for a, b in pairwise(seq):
        super_set.add((a, b))

global_superstates = sorted(super_set)
super_index = {ab: i for i, ab in enumerate(global_superstates)}

print(f"Total de states (1ª ordem): {len(global_states)}")
print(f"Total de superstates (2ª ordem): {len(global_superstates)}")


Total de states (1ª ordem): 32
Total de superstates (2ª ordem): 722


In [4]:

# ============================================================
# 4) Parâmetros
# ============================================================
TOP_K_USERS = None          # defina None para todos os usuários
MIN_SEQ_LEN = 2           # ignora chats com menos que isso
ORDER1_MIN_EDGES = 0      # com eixos fixos, permitimos 0 (vai virar heatmap de zeros)
ORDER2_MIN_EDGES = 0


In [5]:

# ============================================================
# 5) Utilitários para montar matrizes nas ORDENS GLOBAIS
# ============================================================
import numpy as np

def first_order_matrix_global(sequences, global_states, state_index):
    n = len(global_states)
    M = np.zeros((n, n), dtype=int)
    # contar transições
    for seq in sequences:
        for a, b in pairwise(seq):
            if a in state_index and b in state_index:
                M[state_index[a], state_index[b]] += 1
    # normalizar por linha
    row_sums = M.sum(axis=1, keepdims=True)
    with np.errstate(divide='ignore', invalid='ignore'):
        P = M / row_sums
        P = np.nan_to_num(P, nan=0.0)
    edges = int(M.sum())
    return M, P, edges

def second_order_matrix_global(sequences, global_states, state_index, global_superstates, super_index):
    r = len(global_superstates)
    c = len(global_states)
    M = np.zeros((r, c), dtype=int)
    for seq in sequences:
        # precisamos de trios para 2ª ordem, mas as linhas são pares (a,b) globais
        from itertools import tee
        a, b, c_iter = tee(seq, 3)
        next(b, None)
        next(c_iter, None); next(c_iter, None)
        for a1, b1, c1 in zip(a, b, c_iter):
            ab = (a1, b1)
            if ab in super_index and c1 in state_index:
                M[super_index[ab], state_index[c1]] += 1
    row_sums = M.sum(axis=1, keepdims=True)
    with np.errstate(divide='ignore', invalid='ignore'):
        P = M / row_sums
        P = np.nan_to_num(P, nan=0.0)
    edges = int(M.sum())
    return M, P, edges


In [6]:

# ============================================================
# 6) Selecionar usuários e processar
# ============================================================
# Filtrar sequências curtas
seq_by_chat = seq_by_chat[seq_by_chat["sequence"].apply(len) >= MIN_SEQ_LEN].copy()

# Ranking por volume
user_stats = (seq_by_chat.groupby("userId")["sequence"]
                         .agg(chats="count",
                              total_labels=lambda s: int(sum(len(x) for x in s)))
                         .reset_index()
                         .sort_values("total_labels", ascending=False))

if TOP_K_USERS is not None:
    user_stats = user_stats.head(TOP_K_USERS)

user_ids = user_stats["userId"].tolist()
print("Usuários selecionados:", len(user_ids))
user_stats.head(10)


Usuários selecionados: 201


,userId,chats,total_labels
20,117bf560-5011-70e9-de01-b833f5651c83,27,384
194,f16b2520-6021-70ee-b980-7a50d804795d,36,381
101,81fbe5c0-8001-7054-3ac3-3e9db3f6e198,10,377
176,e12ba500-f0e1-70f2-73ab-05c1cbc44b76,40,374
137,b19b0560-c011-70d1-0a4f-c32cfa1ae4cd,15,330
151,c16be560-e061-709d-a447-5a0779910084,37,306
10,01eb0530-d011-70d4-70a6-3d8b36a0733f,30,277
123,a1cb4570-5011-70b5-5fdf-1eaee3de61f4,15,261
92,813b4500-6031-701e-c56f-d60220a6081f,38,260
33,219b2590-e041-704c-6a88-f3a3366c6cc5,17,251


In [7]:

# ============================================================
# 7) Plot fixo por usuário (mesmos eixos para todos)
# ============================================================
summary_rows = []
for i, uid in enumerate(user_ids, start=1):
    user_seqs = seq_by_chat[seq_by_chat["userId"] == uid]["sequence"].tolist()

    # 1ª ordem
    M1, P1, e1 = first_order_matrix_global(user_seqs, global_states, state_index)
    np.savez(os.path.join(out_dir, f"user_{uid}_order1.npz"),
             P=P1, M=M1, states=np.array(global_states, dtype=object), userId=uid, order=1)
    fig = plt.figure(figsize=(10, 8))
    plt.imshow(P1, aspect='auto', vmin=0, vmax=1)
    plt.title(f"User {uid} — Markov 1º ordem (eixos globais)")
    plt.xlabel("Próximo estado")
    plt.ylabel("Estado atual")
    plt.xticks(range(len(global_states)), global_states, rotation=90)
    plt.yticks(range(len(global_states)), global_states)
    plt.colorbar()
    plt.tight_layout()
    out1 = os.path.join(out_dir, f"user_{uid}_order1_fixed.png")
    plt.savefig(out1, dpi=160)
    plt.close(fig)

    # 2ª ordem
    M2, P2, e2 = second_order_matrix_global(user_seqs, global_states, state_index, global_superstates, super_index)
    np.savez(os.path.join(out_dir, f"user_{uid}_order2.npz"),
             P=P2, M=M2,
             states=np.array(global_states, dtype=object),
             superstates=np.array(global_superstates, dtype=object),
             userId=uid, order=2)
    fig = plt.figure(figsize=(12, 9))
    plt.imshow(P2, aspect='auto', vmin=0, vmax=1)
    plt.title(f"User {uid} — Markov 2º ordem (eixos globais)")
    plt.xlabel("Próximo estado")
    plt.ylabel("Par de estados (histórico) — eixos globais")

    # X ticks
    plt.xticks(range(len(global_states)), global_states, rotation=90)
    # Y ticks (subamostrar para legibilidade se houver muitos pares)
    max_ticks = 100
    if len(global_superstates) > max_ticks:
        step = max(1, len(global_superstates) // max_ticks)
        yticks = list(range(0, len(global_superstates), step))
        ylabels = [f"{a}|{b}" for (a,b) in [global_superstates[k] for k in yticks]]
    else:
        yticks = list(range(len(global_superstates)))
        ylabels = [f"{a}|{b}" for (a,b) in global_superstates]

    plt.yticks(yticks, ylabels)
    plt.colorbar()
    plt.tight_layout()
    out2 = os.path.join(out_dir, f"user_{uid}_order2_fixed.png")
    plt.savefig(out2, dpi=160)
    plt.close(fig)

    summary_rows.append({
        "userId": uid,
        "order1_edges": e1,
        "order2_edges": e2,
        "order1_path": out1,
        "order2_path": out2
    })

    if i % 10 == 0:
        print(f"Processados {i}/{len(user_ids)} usuários...")

summary_df = pd.DataFrame(summary_rows)
summary_csv = os.path.join(out_dir, "per_user_fixed_summary.csv")
summary_df.to_csv(summary_csv, index=False, encoding="utf-8")
print("Resumo salvo em:", summary_csv)

summary_df.head(10)


Processados 10/201 usuários...
Processados 20/201 usuários...
Processados 30/201 usuários...
Processados 40/201 usuários...
Processados 50/201 usuários...
Processados 60/201 usuários...
Processados 70/201 usuários...
Processados 80/201 usuários...
Processados 90/201 usuários...
Processados 100/201 usuários...
Processados 110/201 usuários...
Processados 120/201 usuários...
Processados 130/201 usuários...
Processados 140/201 usuários...
Processados 150/201 usuários...
Processados 160/201 usuários...
Processados 170/201 usuários...
Processados 180/201 usuários...
Processados 190/201 usuários...
Processados 200/201 usuários...
Resumo salvo em: artifacts/per_user_fixed_summary.csv


,userId,order1_edges,order2_edges,order1_path,order2_path
0,117bf560-5011-70e9-de01-b833f5651c83,357,330,artifacts/user_117bf560-5011-70e9-de01-b833f56...,artifacts/user_117bf560-5011-70e9-de01-b833f56...
1,f16b2520-6021-70ee-b980-7a50d804795d,345,309,artifacts/user_f16b2520-6021-70ee-b980-7a50d80...,artifacts/user_f16b2520-6021-70ee-b980-7a50d80...
2,81fbe5c0-8001-7054-3ac3-3e9db3f6e198,367,357,artifacts/user_81fbe5c0-8001-7054-3ac3-3e9db3f...,artifacts/user_81fbe5c0-8001-7054-3ac3-3e9db3f...
3,e12ba500-f0e1-70f2-73ab-05c1cbc44b76,334,294,artifacts/user_e12ba500-f0e1-70f2-73ab-05c1cbc...,artifacts/user_e12ba500-f0e1-70f2-73ab-05c1cbc...
4,b19b0560-c011-70d1-0a4f-c32cfa1ae4cd,315,300,artifacts/user_b19b0560-c011-70d1-0a4f-c32cfa1...,artifacts/user_b19b0560-c011-70d1-0a4f-c32cfa1...
5,c16be560-e061-709d-a447-5a0779910084,269,232,artifacts/user_c16be560-e061-709d-a447-5a07799...,artifacts/user_c16be560-e061-709d-a447-5a07799...
6,01eb0530-d011-70d4-70a6-3d8b36a0733f,247,217,artifacts/user_01eb0530-d011-70d4-70a6-3d8b36a...,artifacts/user_01eb0530-d011-70d4-70a6-3d8b36a...
7,a1cb4570-5011-70b5-5fdf-1eaee3de61f4,246,231,artifacts/user_a1cb4570-5011-70b5-5fdf-1eaee3d...,artifacts/user_a1cb4570-5011-70b5-5fdf-1eaee3d...
8,813b4500-6031-701e-c56f-d60220a6081f,222,184,artifacts/user_813b4500-6031-701e-c56f-d60220a...,artifacts/user_813b4500-6031-701e-c56f-d60220a...
9,219b2590-e041-704c-6a88-f3a3366c6cc5,234,217,artifacts/user_219b2590-e041-704c-6a88-f3a3366...,artifacts/user_219b2590-e041-704c-6a88-f3a3366...


In [8]:

# ============================================================
# 8) Exportar ordens globais (para referência/comparabilidade)
# ============================================================
import json
axes_meta = {
    "states_global": global_states,
    "superstates_global": [[a, b] for (a,b) in global_superstates]
}
with open(os.path.join(out_dir, "axes_global.json"), "w", encoding="utf-8") as f:
    json.dump(axes_meta, f, ensure_ascii=False, indent=2)

print("Eixos globais exportados em:", os.path.join(out_dir, "axes_global.json"))


Eixos globais exportados em: artifacts/axes_global.json
